# HW6 — Self-Supervised and Contrastive Representation Learning (STL-10)

**Rajesh Paruchuri** &nbsp;|&nbsp; **SID4 = 1605** &nbsp;|&nbsp; SEED = 1605 &nbsp;|&nbsp; SLICE = 605 &nbsp;|&nbsp; HP_ID = 3 &nbsp;|&nbsp; CLS_A = 5 (dog) &nbsp;|&nbsp; CLS_B = 3 (cat)

Per the standing requirements (Section 0.1), only parameters this homework references are used. HW6 references
**SEED** through the subset sampling, initialisation and query selection. SLICE, HP_ID, CLS_A and CLS_B are reported but
unused: HW6 waives the second hyperparameter model and names no subset start or focus classes.

All parts use the same backbone: **ResNet-18 from `torchvision`, no pretrained weights** (`weights=None`), with the final
`fc` layer replaced by `Identity` so the encoder outputs a 512-d embedding (global-average-pooled `layer4` features).

| Part | What is trained | Data |
|---|---|---|
| A | ResNet-18 + linear head, end-to-end, supervised | 500 labeled images (10% of train, 50 per class) |
| B | ResNet-18 + 4-way rotation head, then frozen encoder + linear probe | 100,000 unlabeled → same 500 labeled |
| C | ResNet-18 + MLP projection head with NT-Xent (SimCLR), then frozen encoder + linear probe | 20,000 unlabeled → same 500 labeled |
| D | Top-5 cosine nearest neighbours in each encoder's test-set embedding space | 8,000 test images |

As the instructions allow, no second hyperparameter configuration is trained, since three models are already compared.

---
## Step 0 — Standing requirements: personal parameters, reproducibility, data check

- **Personal parameters** are derived from SID4 below and printed.
- **Seeds:** Python `random`, NumPy and PyTorch are all seeded with SEED.
- **Logging:** every training and evaluation line goes through `log()`, which prints it and appends it with a timestamp
  to `logs/console.log`. That file is the source of `RUN_LOG.txt`.
- **Checkpoints:** each part saves a checkpoint to `checkpoints/`. Re-running the notebook loads them instead of
  retraining (set `FORCE_RETRAIN = True` to retrain from scratch).

**Exact command for the reported run** (from `hw6/`):
```
.venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=hw6 \
    --ExecutePreprocessor.timeout=-1 ssl_stl10.ipynb
```
**Non-determinism:** the MPS backend does not guarantee bit-identical results for convolution backward passes or
`grid_sample` across runs, even with the same seed. The data subsets, query images and initial weights are exactly
reproducible; final accuracies may differ slightly between runs.

In [ ]:
import os, sys, time, json, math, random, platform, datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.models import resnet18
import matplotlib.pyplot as plt

SID4   = 1605
SEED   = SID4                   # torch.manual_seed, np.random.seed, random
SLICE  = SID4 % 1000            # 605 — not referenced by HW6
HP_ID  = SID4 % 6               # 3   — not referenced by HW6 (second model waived)
CLS_A  = SID4 % 10              # 5   — not referenced by HW6
CLS_B  = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10   # 3 — not referenced by HW6

DATA_DIR, CKPT_DIR, LOG_DIR = "data", "checkpoints", "logs"
for d in (CKPT_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)
SMOKE = os.environ.get("LAB_SMOKE") == "1"   # tiny end-to-end check; full run when unset
FORCE_RETRAIN = False                        # True ignores saved checkpoints
LOG_PATH = os.path.join(LOG_DIR, "console_smoke.log" if SMOKE else "console.log")

def log(*args):
    "Print and append a timestamped line to the console log (source of RUN_LOG.txt)."
    msg = " ".join(str(a) for a in args)
    print(msg)
    stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_PATH, "a") as f:
        for line in msg.splitlines() or [""]:
            f.write(f"[{stamp}] {line}\n")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")

log("=" * 70)
log("RUN_START", datetime.datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z"))
log(f"SID4={SID4}  SEED={SEED}  SLICE={SLICE}  HP_ID={HP_ID}  CLS_A={CLS_A}  CLS_B={CLS_B}")
log("Python     :", sys.version.split()[0])
log("PyTorch    :", torch.__version__, "| torchvision:", torchvision.__version__)
log("Platform   :", platform.platform())
try:
    import subprocess
    cpu = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip()
except Exception:
    cpu = ""
log("Processor  :", cpu or platform.processor() or platform.machine())
log("Device     :", device)
log("SMOKE      :", SMOKE)

In [ ]:
train_ds = torchvision.datasets.STL10(DATA_DIR, split="train", download=True)
test_ds  = torchvision.datasets.STL10(DATA_DIR, split="test", download=True)
unl_ds   = torchvision.datasets.STL10(DATA_DIR, split="unlabeled", download=True)
CLASSES = train_ds.classes

# Keep everything as uint8 tensors (N, 3, 96, 96); batches are moved to the device and augmented there.
X_train, y_train = torch.from_numpy(train_ds.data), torch.from_numpy(train_ds.labels).long()
X_test,  y_test  = torch.from_numpy(test_ds.data),  torch.from_numpy(test_ds.labels).long()
X_unl = torch.from_numpy(unl_ds.data)
if SMOKE:
    X_unl, X_test, y_test = X_unl[:2000], X_test[:1000], y_test[:1000]

log(f"train     : {tuple(X_train.shape)} {X_train.dtype}")
log(f"test      : {tuple(X_test.shape)}")
log(f"unlabeled : {tuple(X_unl.shape)}")
log("classes   :", CLASSES)
log("test class counts:", torch.bincount(y_test, minlength=10).tolist())

**The 10% labeled subset.** 50 images per class are drawn from the 5,000-image train split with a fixed generator, so
the subset is balanced and identical for Parts A, B and C. The 20,000-image SimCLR subset is drawn the same way from the
unlabeled split.

In [ ]:
g = torch.Generator().manual_seed(SEED)
sub_idx = torch.cat([
    (y_train == c).nonzero().squeeze(1)[torch.randperm(int((y_train == c).sum()), generator=g)[:50]]
    for c in range(10)
]).sort().values
X_sub, y_sub = X_train[sub_idx], y_train[sub_idx]

N_SIMCLR = 2000 if SMOKE else 20000
simclr_idx = torch.randperm(len(X_unl), generator=g)[:N_SIMCLR]
X_simclr = X_unl[simclr_idx]

log("labeled subset:", len(X_sub), "images | per class:", torch.bincount(y_sub, minlength=10).tolist())
log("SimCLR subset :", len(X_simclr), "unlabeled images")

# Channel statistics from the unlabeled images only (no labels, no test data).
_x = X_unl[:10000].float() / 255
MEAN = _x.mean(dim=(0, 2, 3)).view(1, 3, 1, 1).to(device)
STD  = _x.std(dim=(0, 2, 3)).view(1, 3, 1, 1).to(device)
del _x
log("mean:", MEAN.flatten().cpu().numpy().round(4), "std:", STD.flatten().cpu().numpy().round(4))

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.4))
for c in range(10):
    for r in range(2):
        img = X_sub[y_sub == c][r]
        axes[r, c].imshow(img.permute(1, 2, 0).numpy())
        axes[r, c].axis("off")
    axes[0, c].set_title(CLASSES[c], fontsize=10)
fig.suptitle("Two examples per class from the 10% labeled subset", y=1.02)
plt.tight_layout(); plt.show()

### Shared helpers

**Augmentations on the GPU.** Applying `torchvision` transforms image by image on the CPU is the bottleneck on a
laptop, so the augmentations are written as batched tensor ops with independent random parameters per image. They match
this `torchvision` pipeline, the four augmentations from the demo:

```python
T.Compose([
    T.RandomResizedCrop(96, scale=(0.2, 1.0)),                    # 1. random resized crop
    T.RandomHorizontalFlip(),                                     # 2. horizontal flip
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),    # 3. color jitter
    T.RandomGrayscale(p=0.2),                                     # 4. grayscale
])
```

The crop and flip are a single affine resampling (`affine_grid` + `grid_sample`). Color jitter changes brightness,
contrast and saturation by blending, and hue by rotating the chroma plane in YIQ space.

In [ ]:
def to_device(xb):
    "uint8 CPU batch -> float [0, 1] on the device"
    return xb.to(device, non_blocking=True).float().div_(255)

def normalize(x):
    return (x - MEAN) / STD

def random_resized_crop_flip(x, scale=(0.2, 1.0), ratio=(3 / 4, 4 / 3), flip_p=0.5):
    B, dev = x.shape[0], x.device
    area = torch.empty(B, device=dev).uniform_(*scale)
    r = torch.exp(torch.empty(B, device=dev).uniform_(math.log(ratio[0]), math.log(ratio[1])))
    w = torch.sqrt(area * r).clamp(max=1.0)          # crop width / image width
    h = torch.sqrt(area / r).clamp(max=1.0)
    cx = (torch.rand(B, device=dev) * 2 - 1) * (1 - w)  # crop centre, normalised coords
    cy = (torch.rand(B, device=dev) * 2 - 1) * (1 - h)
    flip = torch.where(torch.rand(B, device=dev) < flip_p, -1.0, 1.0)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 2] = w * flip, cx
    theta[:, 1, 1], theta[:, 1, 2] = h, cy
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)

def gray(x):
    return 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]

RGB2YIQ = torch.tensor([[0.299, 0.587, 0.114], [0.596, -0.274, -0.322], [0.211, -0.523, 0.312]])
YIQ2RGB = torch.linalg.inv(RGB2YIQ)
RGB2YIQ, YIQ2RGB = RGB2YIQ.to(device), YIQ2RGB.to(device)

def color_jitter(x, b=0.4, c=0.4, s=0.4, hue=0.1, p=0.8):
    B, dev = x.shape[0], x.device
    fac = lambda m: torch.empty(B, 1, 1, 1, device=dev).uniform_(1 - m, 1 + m)
    y = (x * fac(b)).clamp(0, 1)                                     # brightness
    m = gray(y).mean(dim=(2, 3), keepdim=True)
    y = ((y - m) * fac(c) + m).clamp(0, 1)                           # contrast
    gy = gray(y)
    y = ((y - gy) * fac(s) + gy).clamp(0, 1)                         # saturation
    ang = torch.empty(B, device=dev).uniform_(-hue, hue) * 2 * math.pi
    yiq = torch.einsum("ij,bjhw->bihw", RGB2YIQ, y)
    cos, sin = ang.cos().view(B, 1, 1), ang.sin().view(B, 1, 1)
    i, q = yiq[:, 1], yiq[:, 2]
    yiq = torch.stack([yiq[:, 0], cos * i - sin * q, sin * i + cos * q], dim=1)
    y = torch.einsum("ij,bjhw->bihw", YIQ2RGB, yiq).clamp(0, 1)      # hue
    apply = (torch.rand(B, 1, 1, 1, device=dev) < p).float()
    return apply * y + (1 - apply) * x

def random_grayscale(x, p=0.2):
    m = (torch.rand(x.shape[0], 1, 1, 1, device=x.device) < p).float()
    return m * gray(x).expand_as(x) + (1 - m) * x

def simclr_view(x):
    "The four SimCLR augmentations: crop, flip, color jitter, grayscale."
    return random_grayscale(color_jitter(random_resized_crop_flip(x, scale=(0.2, 1.0))))

In [ ]:
def make_encoder():
    "ResNet-18, random init, fc removed -> 512-d embedding."
    m = resnet18(weights=None)
    m.fc = nn.Identity()
    return m

def count_params(m, trainable_only=True):
    return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

def cosine_schedule(opt, total_steps, warmup_steps=0):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        t = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * t))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

@torch.no_grad()
def embed(encoder, X, bs=500):
    "Frozen-encoder embeddings (eval mode, no augmentation) for a uint8 tensor."
    encoder.eval()
    return torch.cat([encoder(normalize(to_device(X[i:i + bs]))).float().cpu() for i in range(0, len(X), bs)])

def load_or_none(name):
    path = os.path.join(CKPT_DIR, name + ("_smoke" if SMOKE else "") + ".pt")
    if os.path.exists(path) and not FORCE_RETRAIN:
        return torch.load(path, map_location="cpu", weights_only=False), path
    return None, path

log("ResNet-18 encoder parameters:", f"{count_params(make_encoder()):,}")

In [ ]:
# Sanity check: the two SimCLR views of the same images
_x = to_device(X_simclr[:6])
_v1, _v2 = simclr_view(_x), simclr_view(_x)
fig, axes = plt.subplots(3, 6, figsize=(10, 5.2))
for j in range(6):
    for r, (t, img) in enumerate([("original", _x), ("view 1", _v1), ("view 2", _v2)]):
        axes[r, j].imshow(img[j].permute(1, 2, 0).cpu().numpy()); axes[r, j].axis("off")
        if j == 0: axes[r, j].set_title(t, loc="left", fontsize=10)
fig.suptitle("SimCLR augmentation check: two independent views per image", y=1.0)
plt.tight_layout(); plt.show()
del _x, _v1, _v2

### Linear evaluation protocol (used in Parts B and C)

The encoder is frozen: `requires_grad_(False)`, `eval()` mode (so BatchNorm statistics don't change), and it is never
passed to the optimizer. Because nothing upstream of the linear layer can change, the 512-d embeddings of the 500 labeled
images and the 8,000 test images are computed once. The `nn.Linear(512, 10)` layer, the only module with trainable
parameters, is then trained on them for 20 epochs. Features are standardized with the mean and std of the **500
training embeddings only**.

Hyperparameters are fixed in advance (Adam, lr 1e-3, weight decay 1e-4, batch 32, 20 epochs). With only 500 labels
there is no separate validation split, and the test set is used **once per model, for the final number only**, never to
choose settings.

In [ ]:
def linear_probe(encoder, epochs=20, lr=1e-3, bs=32, tag=""):
    encoder.requires_grad_(False).eval()
    before = [p.detach().clone() for p in encoder.parameters()]

    f_tr, f_te = embed(encoder, X_sub), embed(encoder, X_test)
    mu, sd = f_tr.mean(0, keepdim=True), f_tr.std(0, keepdim=True) + 1e-6
    f_tr, f_te = (f_tr - mu) / sd, (f_te - mu) / sd

    torch.manual_seed(SEED)
    clf = nn.Linear(512, 10)
    opt = torch.optim.Adam(clf.parameters(), lr=lr, weight_decay=1e-4)
    trainable = sum(p.numel() for p in clf.parameters()) + count_params(encoder)
    log(f"[{tag}] trainable parameters during linear eval: {trainable:,} (encoder: {count_params(encoder)})")

    hist = []
    for ep in range(epochs):
        perm = torch.randperm(len(f_tr))
        tot, correct = 0.0, 0
        for i in range(0, len(f_tr), bs):
            idx = perm[i:i + bs]
            logits = clf(f_tr[idx])
            loss = F.cross_entropy(logits, y_sub[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx); correct += (logits.argmax(1) == y_sub[idx]).sum().item()
        hist.append({"epoch": ep + 1, "loss": tot / len(f_tr), "train_acc": correct / len(f_tr)})
        if ep in (0, 4, 9, 14, epochs - 1):
            log(f"[{tag}] probe epoch {ep + 1:2d}  loss {hist[-1]['loss']:.4f}  train acc {hist[-1]['train_acc']:.3f}")

    assert all(torch.equal(a, b.detach()) for a, b in zip(before, encoder.parameters())), "encoder changed!"
    with torch.no_grad():
        preds = clf(f_te).argmax(1)
        test_acc = (preds == y_test).float().mean().item()
    log(f"[{tag}] encoder unchanged after probe: True | TEST ACCURACY: {test_acc:.4f}")
    return test_acc, hist, preds

---
## Part A — Supervised with limited labels

ResNet-18 + `Linear(512, 10)`, trained end-to-end from random initialisation on the 500 labeled images for **15
epochs**. Light augmentation (random resized crop with scale 0.5–1.0, horizontal flip) is used because 500 images is
very little data for 11M parameters. Optimizer: AdamW (lr 1e-3, weight decay 5e-4), batch 32, cosine schedule.

In [ ]:
class SupervisedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = make_encoder()
        self.head = nn.Linear(512, 10)
    def forward(self, x):
        return self.head(self.encoder(x))

@torch.no_grad()
def evaluate(model, X, y, bs=500):
    model.eval()
    preds = torch.cat([model(normalize(to_device(X[i:i + bs]))).argmax(1).cpu() for i in range(0, len(X), bs)])
    return (preds == y).float().mean().item(), preds

ckpt, path = load_or_none("partA_supervised")
set_seed(SEED)
sup = SupervisedNet().to(device)
if ckpt:
    sup.load_state_dict(ckpt["model"]); histA = ckpt["hist"]; TIME_A = ckpt["train_time"]
    log("Loaded", path); [log(h) for h in histA]
else:
    EPOCHS_A, BS_A = (2 if SMOKE else 15), 32
    opt = torch.optim.AdamW(sup.parameters(), lr=1e-3, weight_decay=5e-4)
    steps = EPOCHS_A * math.ceil(len(X_sub) / BS_A)
    sched = cosine_schedule(opt, steps)
    histA, t0 = [], time.time()
    for ep in range(EPOCHS_A):
        sup.train()
        perm = torch.randperm(len(X_sub))
        tot, correct = 0.0, 0
        for i in range(0, len(X_sub), BS_A):
            idx = perm[i:i + BS_A]
            xb = normalize(random_resized_crop_flip(to_device(X_sub[idx]), scale=(0.5, 1.0)))
            yb = y_sub[idx].to(device)
            logits = sup(xb)
            loss = F.cross_entropy(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item() * len(idx); correct += (logits.argmax(1) == yb).sum().item()
        histA.append({"epoch": ep + 1, "loss": round(tot / len(X_sub), 4), "train_acc": round(correct / len(X_sub), 4)})
        log(histA[-1], f"{time.time() - t0:.0f}s")
    TIME_A = time.time() - t0
    torch.save({"model": sup.state_dict(), "hist": histA, "train_time": TIME_A}, path)

acc_A, preds_A = evaluate(sup, X_test, y_test)
log(f"Part A training time: {TIME_A / 60:.1f} min")
log(f"\nPart A — supervised (500 labels) TEST ACCURACY: {acc_A:.4f}")

---
## Part B — Rotation self-supervised learning

**Pretext task.** Each unlabeled image is rotated by a random k·90°, k ∈ {0, 1, 2, 3}, and the network predicts k
(4-way classification). One random rotation per image per epoch keeps the cost at 100,000 images per epoch. Before
rotating, a mild crop (scale 0.5–1.0) and flip are applied so the encoder cannot memorise exact pixels. Optimizer: SGD
with Nesterov momentum 0.9, lr 0.1, weight decay 5e-4, batch 256, 1-epoch warm-up then cosine, **15 epochs** on all
100,000 unlabeled images.

After pretraining, the rotation head is removed, the encoder is frozen, and the linear probe above is trained on the same
500 labeled images.

In [ ]:
def rotate_batch(x):
    "Rotate each image by a random multiple of 90 degrees; returns images and labels k."
    k = torch.randint(0, 4, (x.shape[0],), device=x.device)
    out = x.clone()
    for r in range(1, 4):
        m = k == r
        if m.any():
            out[m] = torch.rot90(x[m], r, dims=(2, 3))
    return out, k

# Visual check of the pretext task
_x = to_device(X_unl[:1]).repeat(4, 1, 1, 1)
fig, axes = plt.subplots(1, 4, figsize=(7, 2))
for r in range(4):
    axes[r].imshow(torch.rot90(_x[r], r, dims=(1, 2)).permute(1, 2, 0).cpu().numpy())
    axes[r].set_title(f"label {r}: {90 * r}°", fontsize=10); axes[r].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
class RotNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = make_encoder()
        self.rot_head = nn.Linear(512, 4)
    def forward(self, x):
        return self.rot_head(self.encoder(x))

ckpt, path = load_or_none("partB_rotation")
set_seed(SEED)
rot = RotNet().to(device)
if ckpt:
    rot.load_state_dict(ckpt["model"]); histB = ckpt["hist"]; TIME_B = ckpt["train_time"]
    log("Loaded", path); [log(h) for h in histB]
else:
    EPOCHS_B, BS_B = (1 if SMOKE else 15), 256
    steps_per_ep = len(X_unl) // BS_B
    opt = torch.optim.SGD(rot.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)
    sched = cosine_schedule(opt, EPOCHS_B * steps_per_ep, warmup_steps=steps_per_ep)
    prog_path = path.replace(".pt", "_progress.pt")      # per-epoch resume point
    start_ep, histB, elapsed = 0, [], 0.0
    if os.path.exists(prog_path) and not FORCE_RETRAIN:
        prog = torch.load(prog_path, map_location="cpu", weights_only=False)
        rot.load_state_dict(prog["model"]); opt.load_state_dict(prog["opt"]); sched.load_state_dict(prog["sched"])
        start_ep, histB, elapsed = prog["epoch"], prog["hist"], prog["elapsed"]
        torch.set_rng_state(prog["rng"])
        log(f"Resuming after epoch {start_ep} from {prog_path}")
    t0 = time.time() - elapsed
    for ep in range(start_ep, EPOCHS_B):
        rot.train()
        perm = torch.randperm(len(X_unl))
        tot, correct, n = 0.0, 0, 0
        for s in range(steps_per_ep):
            xb = random_resized_crop_flip(to_device(X_unl[perm[s * BS_B:(s + 1) * BS_B]]), scale=(0.5, 1.0))
            xb, k = rotate_batch(xb)
            logits = rot(normalize(xb))
            loss = F.cross_entropy(logits, k)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item() * len(k); correct += (logits.argmax(1) == k).sum().item(); n += len(k)
        histB.append({"epoch": ep + 1, "loss": round(tot / n, 4), "rot_acc": round(correct / n, 4)})
        log(histB[-1], f"{time.time() - t0:.0f}s")
        torch.save({"model": rot.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "epoch": ep + 1,
                    "hist": histB, "elapsed": time.time() - t0, "rng": torch.get_rng_state()}, prog_path)
    TIME_B = time.time() - t0
    torch.save({"model": rot.state_dict(), "hist": histB, "train_time": TIME_B}, path)

In [ ]:
# Remove the rotation head; keep only the encoder and freeze it.
rot_encoder = rot.encoder
del rot.rot_head
log(f"Part B pretraining time: {TIME_B / 60:.1f} min")
acc_B, probeB, preds_B = linear_probe(rot_encoder, tag="Rotation")
log(f"\nPart B — rotation SSL + linear probe TEST ACCURACY: {acc_B:.4f}")

---
## Part C — Contrastive learning (SimCLR-style)

- **Data:** the 20,000-image unlabeled subset from Step 0.
- **Views:** two independent `simclr_view` augmentations per image (crop, flip, color jitter, grayscale).
- **Model:** ResNet-18 encoder → projection head MLP `Linear(512, 512) → BatchNorm → ReLU → Linear(512, 128)`.
- **Loss:** NT-Xent. Projections are L2-normalised so their dot product is the **cosine similarity**, divided by the
  **temperature τ = 0.2**. For a batch of N images (2N views), each view's positive is the other view of the same image
  and the remaining 2N − 2 views are negatives:
  $$\ell_{i,j} = -\log \frac{\exp(\mathrm{sim}(z_i, z_j)/\tau)}{\sum_{k \ne i} \exp(\mathrm{sim}(z_i, z_k)/\tau)}$$
- **Optimizer:** AdamW, lr 1e-3, weight decay 1e-4, batch 256 (512 views), 1-epoch warm-up then cosine, **20 epochs**.

After training, the projection head is discarded, the encoder is frozen, and the same linear probe is applied.

In [ ]:
TAU = 0.2

def nt_xent(z1, z2, tau=TAU):
    z = F.normalize(torch.cat([z1, z2]).float(), dim=1)   # unit vectors -> dot product = cosine similarity
    sim = z @ z.T / tau
    n = z1.shape[0]
    sim.fill_diagonal_(float("-inf"))                     # a view is not its own negative
    targets = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, targets), (sim.argmax(1) == targets).float().mean()

class SimCLR(nn.Module):
    def __init__(self, proj_dim=128):
        super().__init__()
        self.encoder = make_encoder()
        self.projector = nn.Sequential(nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
                                       nn.Linear(512, proj_dim))
    def forward(self, x):
        return self.projector(self.encoder(x))

ckpt, path = load_or_none("partC_simclr")
set_seed(SEED)
simclr = SimCLR().to(device)
log("projection head parameters:", f"{count_params(simclr.projector):,}")
if ckpt:
    simclr.load_state_dict(ckpt["model"]); histC = ckpt["hist"]; TIME_C = ckpt["train_time"]
    log("Loaded", path); [log(h) for h in histC]
else:
    EPOCHS_C, BS_C = (1 if SMOKE else 20), 256
    steps_per_ep = len(X_simclr) // BS_C
    opt = torch.optim.AdamW(simclr.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = cosine_schedule(opt, EPOCHS_C * steps_per_ep, warmup_steps=steps_per_ep)
    prog_path = path.replace(".pt", "_progress.pt")      # per-epoch resume point
    start_ep, histC, elapsed = 0, [], 0.0
    if os.path.exists(prog_path) and not FORCE_RETRAIN:
        prog = torch.load(prog_path, map_location="cpu", weights_only=False)
        simclr.load_state_dict(prog["model"]); opt.load_state_dict(prog["opt"]); sched.load_state_dict(prog["sched"])
        start_ep, histC, elapsed = prog["epoch"], prog["hist"], prog["elapsed"]
        torch.set_rng_state(prog["rng"])
        log(f"Resuming after epoch {start_ep} from {prog_path}")
    t0 = time.time() - elapsed
    for ep in range(start_ep, EPOCHS_C):
        simclr.train()
        perm = torch.randperm(len(X_simclr))
        tot, top1, n = 0.0, 0.0, 0
        for s in range(steps_per_ep):
            x = to_device(X_simclr[perm[s * BS_C:(s + 1) * BS_C]])
            v = normalize(torch.cat([simclr_view(x), simclr_view(x)]))   # both views in one forward pass
            z = simclr(v)
            loss, acc = nt_xent(z[:BS_C], z[BS_C:])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item(); top1 += acc.item(); n += 1
        histC.append({"epoch": ep + 1, "loss": round(tot / n, 4), "pos_top1": round(top1 / n, 4)})
        log(histC[-1], f"{time.time() - t0:.0f}s")
        torch.save({"model": simclr.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "epoch": ep + 1,
                    "hist": histC, "elapsed": time.time() - t0, "rng": torch.get_rng_state()}, prog_path)
    TIME_C = time.time() - t0
    torch.save({"model": simclr.state_dict(), "hist": histC, "train_time": TIME_C}, path)

`pos_top1` is the fraction of views whose most similar other view is their own positive pair, a contrastive accuracy
out of 511 candidates per view (chance ≈ 0.2%).

In [ ]:
# Remove the projection head; keep only the encoder and freeze it.
simclr_encoder = simclr.encoder
del simclr.projector
log(f"Part C pretraining time: {TIME_C / 60:.1f} min")
acc_C, probeC, preds_C = linear_probe(simclr_encoder, tag="SimCLR")
log(f"\nPart C — SimCLR + linear probe TEST ACCURACY: {acc_C:.4f}")

In [ ]:
# Training curves (one panel per model, since the losses are on different scales)
COLORS = {"Supervised": "#2a78d6", "Rotation SSL": "#eb6834", "SimCLR": "#1baf7a"}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
panels = [("Supervised", histA, "loss", "Part A: cross-entropy (10 classes)"),
          ("Rotation SSL", histB, "loss", "Part B: rotation cross-entropy (4-way)"),
          ("SimCLR", histC, "loss", f"Part C: NT-Xent loss (τ = {TAU})")]
for ax, (name, h, key, title) in zip(axes, panels):
    ax.plot([e["epoch"] for e in h], [e[key] for e in h], color=COLORS[name], lw=2, marker="o", ms=4)
    ax.set_title(title, fontsize=11, loc="left"); ax.set_xlabel("epoch"); ax.set_ylabel("training loss")
    ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

---
## Part D — Nearest-neighbour visualisation

The three frozen encoders embed all test images: the Part A supervised encoder (without its classifier head), the
rotation encoder and the SimCLR encoder (both without their pretext heads). Embeddings are L2-normalised, so the dot
product is the **cosine similarity**. For each query, the top-5 most similar *other* test images are retrieved.

Four queries are drawn from four distinct classes with the fixed seed (not hand-picked), and **the same queries are used
for all three encoders**. Neighbour titles are green when the neighbour's class matches the query's and red otherwise.

To go beyond four examples, **precision@5** over every test image is also reported: the fraction of each image's 5
nearest neighbours that share its class, averaged over all queries.

In [ ]:
ENCODERS = {"Supervised": sup.encoder, "Rotation SSL": rot_encoder, "SimCLR": simclr_encoder}
EMB = {name: F.normalize(embed(enc, X_test), dim=1) for name, enc in ENCODERS.items()}

def topk_neighbours(E, q, k=5):
    sims = E @ E[q]
    sims[q] = -float("inf")           # exclude the query itself
    s, idx = sims.topk(k)
    return idx, s

gq = torch.Generator().manual_seed(SEED)
q_classes = torch.randperm(10, generator=gq)[:4]
QUERIES = [int((y_test == c).nonzero().squeeze(1)[torch.randint(int((y_test == c).sum()), (1,), generator=gq)])
           for c in q_classes]
log("query test indices:", QUERIES, "| classes:", [CLASSES[y_test[q]] for q in QUERIES])

In [ ]:
def show_query(q):
    fig, axes = plt.subplots(3, 6, figsize=(12, 6.6))
    qlab = int(y_test[q])
    for r, (name, E) in enumerate(EMB.items()):
        idx, s = topk_neighbours(E, q)
        axes[r, 0].imshow(X_test[q].permute(1, 2, 0).numpy())
        axes[r, 0].set_title(f"QUERY\n{CLASSES[qlab]}", fontsize=10, fontweight="bold")
        axes[r, 0].set_ylabel(name, fontsize=11, fontweight="bold")
        for j, (i, sv) in enumerate(zip(idx.tolist(), s.tolist())):
            lab = int(y_test[i])
            axes[r, j + 1].imshow(X_test[i].permute(1, 2, 0).numpy())
            axes[r, j + 1].set_title(f"{CLASSES[lab]}\ncos {sv:.3f}", fontsize=9,
                                     color="#008300" if lab == qlab else "#c0392b")
        for ax in axes[r]:
            ax.set_xticks([]); ax.set_yticks([])
        hits = sum(int(y_test[i]) == qlab for i in idx.tolist())
        axes[r, 5].text(1.05, 0.5, f"{hits}/5\nsame class", transform=axes[r, 5].transAxes, va="center", fontsize=10)
    fig.suptitle(f"Test image #{q} ({CLASSES[qlab]}): top-5 cosine nearest neighbours per encoder", y=1.0)
    plt.tight_layout(); plt.show()

for q in QUERIES:
    show_query(q)

In [ ]:
@torch.no_grad()
def precision_at_k(E, y, k=5, chunk=1000):
    hits = 0
    for i in range(0, len(E), chunk):
        sims = E[i:i + chunk] @ E.T
        sims[torch.arange(sims.shape[0]), torch.arange(i, i + sims.shape[0])] = -float("inf")
        nn_idx = sims.topk(k, dim=1).indices
        hits += (y[nn_idx] == y[i:i + chunk, None]).sum().item()
    return hits / (len(E) * k)

P5 = {name: precision_at_k(E, y_test) for name, E in EMB.items()}
ACC = {"Supervised": acc_A, "Rotation SSL": acc_B, "SimCLR": acc_C}
query_hits = {name: [sum(int(y_test[i]) == int(y_test[q]) for i in topk_neighbours(E, q)[0].tolist()) for q in QUERIES]
              for name, E in EMB.items()}

log(f"{'Model':<14}{'Test accuracy':>15}{'Precision@5 (all test)':>25}{'Same-class hits on queries':>30}")
for name in ACC:
    log(f"{name:<14}{ACC[name]:>15.4f}{P5[name]:>25.4f}{str(query_hits[name]):>30}")
log("(chance level for both metrics: 0.10)")

TIMES = {"Supervised": TIME_A, "Rotation SSL": TIME_B, "SimCLR": TIME_C}
for name in ACC:
    log(f"{name:<14} training time: {TIMES[name] / 60:.1f} min")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
names = list(ACC)
for ax, (metric, vals) in zip(axes, [("Linear / end-to-end test accuracy", ACC), ("kNN precision@5 on test set", P5)]):
    bars = ax.bar(names, [vals[n] for n in names], color=[COLORS[n] for n in names], width=0.55)
    ax.bar_label(bars, labels=[f"{vals[n]:.3f}" for n in names], padding=3, fontsize=10)
    ax.axhline(0.1, color="#888", lw=1, ls="--"); ax.text(2.35, 0.105, "chance", fontsize=9, color="#666", ha="right")
    ax.set_title(metric, loc="left", fontsize=11); ax.set_ylim(0, max(vals.values()) * 1.25)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

---
## Failure and error analysis

### 1. Which classes each model gets wrong

Per-class test accuracy and confusion matrices for the three classifiers: the Part A network, and the Part B/C linear
probes. Rows are true classes, columns are predictions; each row is normalised so its cells sum to 1.

In [ ]:
from collections import Counter
PREDS = {"Supervised": preds_A, "Rotation SSL": preds_B, "SimCLR": preds_C}
CONF = {}
for name, p in PREDS.items():
    cm = torch.zeros(10, 10)
    for t, q in zip(y_test.tolist(), p.tolist()):
        cm[t, q] += 1
    CONF[name] = cm / cm.sum(1, keepdim=True)

PER_CLASS = {name: CONF[name].diag().tolist() for name in PREDS}
log(f"{'class':<10}" + "".join(f"{n:>15}" for n in PREDS))
for c in range(10):
    log(f"{CLASSES[c]:<10}" + "".join(f"{PER_CLASS[n][c]:>15.3f}" for n in PREDS))

fig, axes = plt.subplots(1, 3, figsize=(17, 5.4))
for ax, (name, cm) in zip(axes, CONF.items()):
    im = ax.imshow(cm.numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=60, fontsize=9)
    ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES, fontsize=9)
    for i in range(10):
        for j in range(10):
            v = cm[i, j].item()
            if v >= 0.1:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if v > 0.5 else "#222")
    ax.set_title(f"{name}  (acc {ACC[name]:.3f})", loc="left", fontsize=11)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.colorbar(im, ax=axes, fraction=0.015, label="fraction of true class")
plt.show()

TOP_CONFUSIONS = {}
for name, cm in CONF.items():
    off = cm.clone(); off.fill_diagonal_(0)
    vals, idx = off.flatten().topk(3)
    TOP_CONFUSIONS[name] = [(CLASSES[i // 10], CLASSES[i % 10], round(v, 3)) for v, i in zip(vals.tolist(), idx.tolist())]
    log(f"{name:<14} most frequent confusions (true -> predicted, rate):", TOP_CONFUSIONS[name])

### 2. Retrieval failures

A retrieval *failure* is a test query for which **none** of the top-5 cosine neighbours shares its class. The cell
below counts them for each encoder. It then shows three failures per encoder: the queries whose neighbours are most
confidently wrong (highest mean cosine similarity among 0/5 queries). Those reveal what the embedding treats as "similar".

In [ ]:
@torch.no_grad()
def per_query_hits(E, y, k=5, chunk=1000):
    out = []
    for i in range(0, len(E), chunk):
        sims = E[i:i + chunk] @ E.T
        sims[torch.arange(sims.shape[0]), torch.arange(i, i + sims.shape[0])] = -float("inf")
        s, nn_idx = sims.topk(k, dim=1)
        out.append(torch.stack([(y[nn_idx] == y[i:i + chunk, None]).sum(1).float(), s.mean(1)], 1))
    return torch.cat(out)

FAIL = {}
for name, E in EMB.items():
    h = per_query_hits(E, y_test)
    zero = (h[:, 0] == 0).nonzero().squeeze(1)
    FAIL[name] = {"n_zero": len(zero), "frac_zero": len(zero) / len(E),
                  "worst": zero[h[zero, 1].argsort(descending=True)[:3]].tolist(),
                  "by_class": Counter(CLASSES[y_test[i]] for i in zero.tolist()).most_common(3)}
    log(f"{name:<14} queries with 0/5 same-class neighbours: {len(zero):5d} ({len(zero) / len(E):.1%}) "
          f"| most affected classes: {FAIL[name]['by_class']}")

rows = [(name, q) for name in EMB for q in FAIL[name]["worst"]]
fig, axes = plt.subplots(len(rows), 6, figsize=(11, 2.05 * len(rows)))
for r, (name, q) in enumerate(rows):
    idx, s = topk_neighbours(EMB[name], q)
    axes[r, 0].imshow(X_test[q].permute(1, 2, 0).numpy())
    axes[r, 0].set_title(f"query: {CLASSES[y_test[q]]}", fontsize=9, fontweight="bold")
    axes[r, 0].set_ylabel(name, fontsize=9, fontweight="bold")
    for j, (i, sv) in enumerate(zip(idx.tolist(), s.tolist())):
        axes[r, j + 1].imshow(X_test[i].permute(1, 2, 0).numpy())
        axes[r, j + 1].set_title(f"{CLASSES[y_test[i]]} {sv:.2f}", fontsize=8, color="#c0392b")
    for ax in axes[r]:
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Most confident retrieval failures (0/5 neighbours share the query's class)", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
results = {"SID4": SID4, "SEED": SEED, "test_accuracy": ACC, "precision_at_5": P5,
           "train_time_min": {n: round(t / 60, 2) for n, t in TIMES.items()},
           "query_indices": QUERIES, "query_hits": query_hits, "per_class_accuracy": PER_CLASS,
           "top_confusions": TOP_CONFUSIONS,
           "retrieval_failures": {n: {k: v for k, v in f.items()} for n, f in FAIL.items()},
           "histories": {"A": histA, "B": histB, "C": histC, "probeB": probeB, "probeC": probeC}}
with open("results" + ("_smoke" if SMOKE else "") + ".json", "w") as f:
    json.dump(results, f, indent=2)
log("RUN_END", datetime.datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z"))

---
## Analysis

*(Written after the full run. All numbers below come from the outputs above.)*

ANALYSIS_PLACEHOLDER